#### 1.paths & URL

In [ ]:
# Intent: set project paths and the official RBNZ M10 download URL.
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # if kernel cwd is notebooks/

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = RAW_DIR / "rbnz_m10_housing.csv"

# Official M10 Excel (1990–current). RBNZ does not publish a direct CSV for this series.
M10_URL = (
    "https://www.rbnz.govt.nz/-/media/project/sites/rbnz/"
    "files/statistics/series/m/m10/hm10.xlsx"
)

#### 2. download bytes (URL or local fallback)


In [ ]:
# Intent: download M10 from URL, or read a manually placed file under data/raw/.
import io
from typing import Tuple

import requests

LOCAL_CANDIDATES = [
    RAW_DIR / "hm10.xlsx",
    RAW_DIR / "rbnz_m10_housing.xlsx",
    RAW_DIR / "rbnz_m10_housing.csv",
]


def fetch_bytes(url: str, timeout: int = 60) -> Tuple[bytes, str]:
    """Download URL content; return (bytes, suggested_extension)."""
    resp = requests.get(url, timeout=timeout)
    resp.raise_for_status()
    ext = ".csv"
    ctype = (resp.headers.get("Content-Type") or "").lower()
    if "spreadsheet" in ctype or url.lower().endswith((".xlsx", ".xls")):
        ext = ".xlsx"
    elif url.lower().endswith(".csv") or "text/csv" in ctype:
        ext = ".csv"
    return resp.content, ext


def load_local_or_url(url: str) -> Tuple[bytes, str, str]:
    """Prefer local manual download; else fetch from URL."""
    for path in LOCAL_CANDIDATES:
        if path.exists():
            return path.read_bytes(), path.suffix.lower(), f"local:{path.name}"
    content, ext = fetch_bytes(url)
    return content, ext, f"url:{url}"


In [6]:
raw_bytes, file_ext, source_label = load_local_or_url(M10_URL)
print(f"Loaded from {source_label}, detected extension: {file_ext}")

Loaded from local:hm10.xlsx, detected extension: .xlsx


#### 3.read CSV or Excel

In [ ]:
# Intent: parse RBNZ file whether it arrives as CSV or Excel.
from typing import Union

import pandas as pd


def read_rbnz_table(content: bytes, ext: str) -> pd.DataFrame:
    """Read RBNZ M10 as DataFrame from CSV or Excel bytes."""
    if ext == ".csv":
        return pd.read_csv(io.BytesIO(content), encoding="utf-8")
    if ext in (".xlsx", ".xls"):
        xls = pd.ExcelFile(io.BytesIO(content), engine="openpyxl")
        print("Sheet names:", xls.sheet_names)
        # M10 is usually one main sheet; adjust sheet_name if inspect shows otherwise.
        return pd.read_excel(xls, sheet_name=0, header=None)
    raise ValueError(f"Unsupported extension: {ext}")


df_raw = read_rbnz_table(raw_bytes, file_ext)
df_raw.head(15)

#### 4.inspect structure

In [ ]:
# Intent: understand shape, dtypes, and column layout before cleaning.
print("Shape:", df_raw.shape)
print("\n--- dtypes ---")
print(df_raw.dtypes)
print("\n--- first 20 rows ---")
display(df_raw.head(20))
print("\n--- info ---")
df_raw.info()